### Subtraction with Two's Complement Representation in SuperNeuroMAT

**Tutorial Author and Code Implementation:** Simon Weston <br> 
**Algorithmic Design:** Simon Weston (independent two's complement modifications) <br>
**Base Structural Influence:** *Virtual Neuron: A Neuromorphic Approach for Encoding Numbers* (Date et al., 2023) <br>

In this tutorial we'll build on our existing ripple-carry adder (see tutorial #12) to create a generalized spiking neural network that subtracts two numbers. The system takes in two n-digit binary inputs and produces a binary output, using spikes to represent 1's and the absence of spikes to represent 0's. The highest output bit spikes when negative outputs are produced. 

#### System Structure

![twos complement subtraction system diagram](img/twos_complement_subtraction_images/subtraction_diagram.jpg)

**Diagram 1:** Modified by Simon Weston, base structure from *Virtual Neuron: A Neuromorphic Approach for Encoding Numbers* (Date et al., 2023)

The subtraction system is illustrated in the diagram above. <br>
*bit_count in this system refers to the # of output bits. The range of the three-bit output in the system above is [-4, 3]. Therefore, our inputs must each have one less bit, so that all possible differences, [-3, 3] for two bits, can be represented*

<br>
Two's complementation encoding of negative numbers allows us to subtract by through addition of a negative. Negative numbers in two's complement are represented by inverting (0's become 1's, 1's become 0's) their positive counterpart and adding 1 (-x = invert(x) + 1).  

There are only 3 changes to the original ripple-adder architecture (highlighted): <br>

1: A new set of neurons representing an input's two's complement, are added, consisting of the inverse and an add_one neuron <br>
2: A neuron of threshold 2 in bit 0 is added (since there's now 3 possible spikes in bit 0) <br>
3: The highest output bit now represents the sign of the output <br>

#### Set Up / Change #1: Creating/Connecting Two's Complement Neurons

In [122]:
#If you haven't yet installed SuperNeuroMat, copy the following and run it in your terminal: 
#pip install superneuromat

Let's begin by taking a copy of our addition structure (see 12_ripple_carry_adder_tutorial) for reference, and renaming it. As we make changes to it, feel free to comment out the obsolete regions so that all code runs. 

In [ ]:
import superneuromat as snm

SUBTRACT = snm.SNN()

# set placeholder bit_count (we will move it to a dedicated "test" section later):
bit_count = 3

x_input = []
y_input = []
bit_groups = []
z_outputs = []

# create inputs
for i in range(bit_count):
    x_input.append(SUBTRACT.create_neuron(threshold=0))

for i in range(bit_count):
    y_input.append(SUBTRACT.create_neuron(threshold=0))


# create hidden layer
for bit in range(bit_count):

    cluster = []

    if bit == 0:
        thresholds = [0, 1]
    else:
        thresholds = [0, 1, 2]

    for threshold in thresholds:
        neuron = SUBTRACT.create_neuron(threshold=threshold)
        cluster.append(neuron)

    bit_groups.append(cluster)

        

# create output neurons
for i in range(bit_count + 1):
    z_outputs.append(SUBTRACT.create_neuron(threshold=0))

# input to hidden layer synapses
for bit in range(bit_count):
    
    delay = bit + 1
    
    for neuron in bit_groups[bit]:
        SUBTRACT.create_synapse(
                pre_id = x_input[bit],
                post_id = neuron,
                weight = 1.0,
                delay = delay
            )
        SUBTRACT.create_synapse(
                pre_id = y_input[bit],
                post_id = neuron,
                weight = 1.0,
                delay = delay
            )
        
# carry over synapses
for bit in range(bit_count - 1):

    carry_neuron = bit_groups[bit][1]

    for neuron in bit_groups[bit + 1]:
        SUBTRACT.create_synapse(
            carry_neuron,
            neuron,
            weight = 1.0,
            delay = 1
        )

# hidden to output layer synapses
for bit in range(bit_count):
    
    delay = bit_count - bit
            
    threshold_0 = bit_groups[bit][0]
    threshold_1 = bit_groups[bit][1]

    SUBTRACT.create_synapse(
        pre_id = threshold_0,
        post_id = z_outputs[bit],
        weight = 1.0,
        delay = delay
    )

    SUBTRACT.create_synapse(
        pre_id = threshold_1,
        post_id = z_outputs[bit],
        weight = -1.0,
        delay = delay
    )

    if bit > 0:

        threshold_2 = bit_groups[bit][2]

        SUBTRACT.create_synapse(
            pre_id = threshold_2,
            post_id = z_outputs[bit],
            weight = 1.0,
            delay = delay
        )

# final carry to final output
final_carry = bit_groups[-1][1]

SUBTRACT.create_synapse(
    pre_id = final_carry,
    post_id = z_outputs[-1],
    weight = 1.0,
    delay = 1
)

<Synapse 26 on SNN at 0x1f8344a2900>

Now let's create an additional layer of neurons, x_inverted, that will hold x input's two's complement representation. 

We want all neurons to spike unless they're inhibited by spikes from x_input, so our reset_state (1/2) > threshold (0). We also want them to only spike once, so refractory_period must be huge (1000 timesteps). Lastly, to prevent x_inverted from spiking at timestep 0 before spikes from x_input have arrived, we set initial_state = -3/2 so activation leaks to -1/2 at t=0 (< threshold) and 1/2 at t=1 (> threshold, spikes unless inhibited by x_input).

In [124]:
x_inverted = []

for i in range(bit_count-1):
    neuron = SUBTRACT.create_neuron(
        threshold=0,
        leak=1,
        reset_state=1/2,
        initial_state=-3/2,
        refractory_period=1000
    )
    x_inverted.append(neuron)

Creating synapses connecting them to the original input:

In [125]:
for i in range(bit_count):
    SUBTRACT.create_synapse(
        x_input[i],
        x_inverted[i],
        weight=-1000,
    )

IndexError: list index out of range

Now we'll edit the input to hidden layer synapses section to take input from x_inverted instead of x_input, and adjust delays accordingly. Now we want all neurons to arrive to their middle layer bit at t = bit + 2 (our first adding operations now occur at t = 2 instead of t = 1) to accomodate the time step it takes to convert x_input --> x_inverted.

In [ ]:
# from input to hidden layer synapses section:

for bit in range(bit_count):
    
    for neuron in bit_groups[bit]:
        SUBTRACT.create_synapse(
            pre_id = x_inverted[bit], # changed from x_input[bit]
            post_id = neuron,
            weight = 1.0,
            delay = bit + 1           # adding the time step for conversion, our first inputs will enter at the adder at t=2.
        )

        SUBTRACT.create_synapse(      # this line will produce an error because we've created these synapses already
            pre_id = y_input[bit],    # in our copy of the addition structure, comment out the creation of these synapses to make this run
            post_id = neuron,
            weight=1.0,
            delay = bit + 2           # as x_inputs --> x_inverted, y_input is waiting an extra timestep inside the synapse to enter the adder in sync at t=2.
        )


RuntimeError: Synapse already exists: <Synapse id: 1 | pre: 2 | post: 4 | weight: 1 | delay:   1 | stdp disabled>If this was intentional, choose arg exist=<'dontadd', 'overwrite'>.

We will create our add_one neuronr in the next section for greater convenience.

#### Change #2: Adding a threshold-2 neuron to bit 0 and the add_one neuron

This is a pretty simple change. In our hidden layer creation section, we can now set all clusters to have neurons with thresholds [0, 1, 2]:

In [ ]:
# create hidden layer
for bit in range(bit_count):

    cluster = []

#    original:
#    if bit == 0:
#        thresholds = [0, 1]
#    else:
#        thresholds = [0, 1, 2]


#   subtraction system (a simplification):
    thresholds = [0, 1, 2]

    for threshold in thresholds:
        neuron = SUBTRACT.create_neuron(threshold=threshold)
        cluster.append(neuron)

    bit_groups.append(cluster)


Creating add_one and connecting it to bit 0:

In [ ]:
add_one = SUBTRACT.create_neuron(threshold=0)

for neuron in bit_groups[0]:
    SUBTRACT.create_synapse(add_one, neuron, delay = 2) # inputs reach middle layer at t = 2 as mentioned earlier

SUBTRACT.add_spike(0, add_one)

#### Change #3: Changing output to represent signed result

Originally, our network had bit_count + 1 output neurons because there was the possibility of overflow when adding two bit_count values. Since overflow is irrelevant to interpreting our two's complement output, there will be one less output neuron (bit_count), with the highest bit signalng the sign of the output. 

In [ ]:
# create output neurons
for i in range(bit_count):   # changed from bit_count + 1
    z_outputs.append(SUBTRACT.create_neuron(threshold=0))

Since we no longer have a bit_count + 1 output neuron, we don't need the final carry synapses:

In [ ]:
"""
# final carry to final output
final_carry = bit_groups[-1][1]

SUBTRACT.create_synapse(
    pre_id = final_carry,
    post_id = z_outputs[-1],
    weight = 1.0,
    delay = 1
)
"""

#### Step 4: Using a function and interpreting results 

Now we'll add our test, simulate, and decoding sections, all of which are nearly identical to that of the addition network. We simulate for an additional timestep (bit_count + 3 rather than bit_count + 2) because we need time to invert our x input. Our decoding is very similar to that of the addition network, except we subtract a value of 2 ** highest bit from the result to align with the two's complement representation.

Feel free to experiment with different inputs to confirm the network's accuracy.

In [1]:
import superneuromat as snm

def twos_complement_subtraction(bit_count):

    TCS = snm.SNN() # stands for two's complement subtraction


    x_inverted = []
    x_input = []
    y_input = []
    bit_groups = []
    z_outputs = []

    # create inputs and twos complement representation of x
    for i in range(bit_count):
        x_input.append(TCS.create_neuron(threshold=0))

    for i in range(bit_count):
        y_input.append(TCS.create_neuron(threshold=0))

    for i in range(bit_count):
        neuron = TCS.create_neuron(
            threshold=0,
            leak=1,
            reset_state=1/2,
            initial_state=-3/2,
            refractory_period=1000
        )
        x_inverted.append(neuron)

    add_one = TCS.create_neuron(threshold=0)


    # create hidden layer
    for bit in range(bit_count):

        cluster = []

        thresholds = [0, 1, 2]

        for threshold in thresholds:
            neuron = TCS.create_neuron(threshold=threshold)
            cluster.append(neuron)

        bit_groups.append(cluster)


    # create output neurons
    for i in range(bit_count):
        z_outputs.append(TCS.create_neuron(threshold=0))

    # input to inverted synapses
    for i in range(bit_count):
        TCS.create_synapse(
            x_input[i],
            x_inverted[i],
            weight=-1000,
        )
        
    # input to hidden layer synapses
    for bit in range(bit_count):
        
        for neuron in bit_groups[bit]:
            TCS.create_synapse(
                pre_id = x_inverted[bit],
                post_id = neuron,
                weight = 1.0,
                delay = bit + 1          
            )

            TCS.create_synapse(      
                pre_id = y_input[bit],    
                post_id = neuron,
                weight=1.0,
                delay = bit + 2          
            )

    for neuron in bit_groups[0]:
            TCS.create_synapse(add_one, neuron, delay = 2)

            
    # carry over synapses
    for bit in range(bit_count - 1):

        carry_neuron = bit_groups[bit][1]

        for neuron in bit_groups[bit + 1]:
            TCS.create_synapse(
                carry_neuron,
                neuron,
                weight = 1.0,
                delay = 1
            )

    # hidden to output layer synapses
    for bit in range(bit_count):
        
        delay = bit_count - bit
                
        threshold_0 = bit_groups[bit][0]
        threshold_1 = bit_groups[bit][1]
        threshold_2 = bit_groups[bit][2]

        TCS.create_synapse(
            pre_id = threshold_0,
            post_id = z_outputs[bit],
            weight = 1.0,
            delay = delay
        )

        TCS.create_synapse(
            pre_id = threshold_1,
            post_id = z_outputs[bit],
            weight = -1.0,
            delay = delay
        )

        TCS.create_synapse(
            pre_id = threshold_2,
            post_id = z_outputs[bit],
            weight = 1.0,
            delay = delay
        )

    return TCS, x_input, y_input, z_outputs, bit_groups, add_one




### Test Section:


BIT_COUNT = 4

TCS, x_input, y_input, z_outputs, bit_groups, add_one = twos_complement_subtraction(BIT_COUNT)

y_value = 5      # difference = y - x
x_value = 2


# spiking inputs:
TCS.add_spike(0, add_one)

for bit in range(BIT_COUNT):

    if (x_value >> bit) & 1:
        TCS.add_spike(0, x_input[bit])

    if (y_value >> bit) & 1:
        TCS.add_spike(0, y_input[bit])

# simulate: 
simulate_time = BIT_COUNT + 3
TCS.simulate(simulate_time)


### Decoding Section: 

# fnding output in binary
output_bits = []

for neuron in z_outputs:
    neuron_id = neuron.idx

    spiked = any(
        TCS.spike_train[t][neuron_id]
        for t in range(len(TCS.spike_train))
    )

    if spiked:
        output_bits.append(1)
    else:
        output_bits.append(0)

binary_output = output_bits[::-1]

print("\nOutput bits:")
print(binary_output)


# converting binary output to base-ten:
result = 0

for bit in binary_output:
    result = result * 2 + bit

# interpreting two's complement output:
if binary_output[0] == 1:
    result -= 2 ** len(binary_output)

print("Expected:", y_value - x_value)
print("\nResult:", result)

print("\nSpike train:")
TCS.print_spike_train()



if result == y_value - x_value:
        print("Correct")
else:
        print("Incorrect")


Output bits:
[0, 1, 0]
Expected: 2

Result: 2

Spike train:
t|id0 1 2 3 4 5 6 7 8 9 101112131415161718192021222324252627282930313233343536373839404142434445464748495051525354555657585960 
0: [├─│ │ ├─├─│ │ │ │ ├─│ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ ]
1: [│ │ │ │ │ │ │ ├─├─│ │ │ │ │ │ │ │ │ │ │ │ │ ├─├─├─│ ├─│ │ ├─│ │ ├─│ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ ├─├─├─│ │ │ │ │ │ │ │ │ ]
2: [│ │ │ │ │ │ │ │ │ │ ├─├─│ │ │ │ │ │ │ │ │ │ │ │ │ ├─│ ├─├─│ ├─├─│ ├─├─│ │ │ │ ├─│ │ │ │ ├─│ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ ]
3: [│ │ │ │ │ │ │ │ │ │ │ │ │ ├─├─├─│ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ ├─│ │ │ │ ├─│ │ │ │ ├─│ │ │ │ │ │ ├─│ ├─│ │ │ │ │ │ ]
4: [│ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ ├─├─│ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ ├─│ ├─│ │ ├─├─├─]
5: [│ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ ├─│ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ ]
Correct


Congratulations, you have completed this tutorial!